# Pure Softmax Scalability Benchmark

### Architecture: 2-Pass Streaming
- **Pass 1**: Block-scan max + sum_exp (pre-scan eliminates per-beat rescaling)
- **Pass 2**: Normalize (exp2 × recip)  AXI-S output
- **Bus**: 128-bit AXI-Stream, 8 elements/beat, 333 MHz, 0 DSP, 2 BRAM

In [1]:
from IPython.display import display, HTML

display(HTML('''
<style>
    /* Increase output cell size */
    .jp-Cell-outputArea, 
    .output_wrapper, 
    .output_area,
    .jp-OutputArea {
        max-width: 100% !important;
        width: 100% !important;
    }
    
    /* Increase content size */
    .jp-OutputArea-child {
        max-width: 100% !important;
        width: 100% !important;
    }
    
    /* Increase notebook container size */
    .jp-Notebook, 
    .container {
        width: 95% !important;
        max-width: 95% !important;
    }
    
    /* Increase rendered HTML size */
    .jp-RenderedHTMLCommon {
        max-width: 100% !important;
    }
</style>
'''))

In [2]:
# =========================================================================
# Configuration and Hardware Setup
# =========================================================================
import os, time, csv, json, datetime
import numpy as np
import torch
import torch.nn.functional as F
from pynq import Overlay, allocate, MMIO
import matplotlib.pyplot as plt

RUN_MODE = 'A'  # 'A' = Early-Stop, 'B' = Full-Run

BITSTREAM    = 'my_softmax15.bit'
IP_NAME      = 'softmax_dl2_v6_0'
DMA_NAME     = 'axi_dma_0'
IP_FCLK_HZ   = 333_000_000
AXIS_WIDTH   = 128
IN_FRAC      = 12; OUT_FRAC = 15
K_SCHEDULE = [1024, 2048, 4096, 8192, 16384, 32768]
N_SAMPLES_PER_SCENARIO = 200

# Pass thresholds
THRESH = {
    'ConfArgMatch': 99.0,
    'MAE':          1e-3,
    'Cosine':       0.999,
    'KL':           0.5,
    'Valid':        99.0,
    'HW_Errors':    0,
}
CONF_GAP_THRESH = 2.0 / (2**OUT_FRAC)

print(f'Loading: {BITSTREAM}...')
overlay = Overlay(BITSTREAM)
dma = getattr(overlay, DMA_NAME)
dma_send = dma.sendchannel
dma_recv = dma.recvchannel
mmio = MMIO(overlay.ip_dict[IP_NAME]['phys_addr'], 0x10000)
DMA_MAX_BYTES = dma_send._max_size
DMA_MAX_K = (DMA_MAX_BYTES // 16) * 8
print(f'DMA max transfer: {DMA_MAX_BYTES:,} bytes to max K = {DMA_MAX_K:,}')
original_schedule = K_SCHEDULE.copy()
K_SCHEDULE = [k for k in K_SCHEDULE
              if ((k + 7) // 8) * 8 * 2 <= DMA_MAX_BYTES]
if len(K_SCHEDULE) < len(original_schedule):
    skipped = [k for k in original_schedule if k not in K_SCHEDULE]
    print(f'DMA limit: Skipping K={skipped} '
          f'(exceed {DMA_MAX_BYTES} bytes)')
    print(f'    Fix: Vivado to DMA IP to '
          f'Buffer Length Register Width to 23 bits')

# === v6.0 Register Map ===
REG_CTRL        = 0x00
REG_STATUS      = 0x04
REG_ARGMAX_IDX  = 0x08
REG_ARGMAX_VAL  = 0x0C
REG_TOTAL_LO    = 0x10; REG_TOTAL_HI   = 0x14
REG_P1_LO       = 0x18; REG_P1_HI      = 0x1C
REG_P2_LO       = 0x20; REG_P2_HI      = 0x24
REG_STALL_LO    = 0x28; REG_STALL_HI   = 0x2C
REG_K_CONFIG    = 0x30
REG_COMPUTE_LO  = 0x34; REG_COMPUTE_HI = 0x38

CTRL_START_P1=0x01; CTRL_START_P2=0x02
CTRL_CLEAR_PERF=0x04; CTRL_CLEAR_ERR=0x08
STATUS_P1_DONE=0x01; STATUS_P2_DONE=0x02
STATUS_BUSY=0x04; STATUS_ERROR=0x08

def rd64(lo, hi):
    return (mmio.read(hi) << 32) | mmio.read(lo)

run_str = 'Early-Stop' if RUN_MODE == 'A' else 'Full-Run'
print(f'Run Mode: {RUN_MODE} — {run_str}')
print(f'K schedule: {K_SCHEDULE}')
print(f'Scenarios: 7 × {N_SAMPLES_PER_SCENARIO} '
      f'= {7*N_SAMPLES_PER_SCENARIO} per K')

Loading: my_softmax15.bit...


DMA max transfer: 8,388,607 bytes to max K = 4,194,296
Run Mode: A — Early-Stop
K schedule: [1024, 2048, 4096, 8192, 16384, 32768]
Scenarios: 7 × 200 = 1400 per K


In [3]:
# =========================================================================
# Core Functions
# =========================================================================
EPS = 1e-12

def kl_div(p, q):
    p, q = np.clip(p, EPS, 1), np.clip(q, EPS, 1)
    return float(np.sum(p * np.log(p / q)))

def cos_sim(a, b):
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / d) if d > 0 else 0

def pack_128b(int16_arr, k_padded):
    n_beats = k_padded // 8
    u = int16_arr.view(np.uint16).astype(np.uint32)
    r = np.empty(n_beats * 4, dtype=np.uint32)
    for b in range(n_beats):
        s = b * 8
        r[b*4+0] = u[s]   | (u[s+1] << 16)
        r[b*4+1] = u[s+2] | (u[s+3] << 16)
        r[b*4+2] = u[s+4] | (u[s+5] << 16)
        r[b*4+3] = u[s+6] | (u[s+7] << 16)
    return r

def unpack_128b(w32, k_actual, k_padded):
    n_beats = k_padded // 8
    p = np.empty(k_padded, dtype=np.uint16)
    for b in range(n_beats):
        w = w32[b*4:b*4+4].astype(np.uint32); s = b * 8
        p[s]   = w[0] & 0xFFFF;       p[s+1] = (w[0] >> 16) & 0xFFFF
        p[s+2] = w[1] & 0xFFFF;       p[s+3] = (w[1] >> 16) & 0xFFFF
        p[s+4] = w[2] & 0xFFFF;       p[s+5] = (w[2] >> 16) & 0xFFFF
        p[s+6] = w[3] & 0xFFFF;       p[s+7] = (w[3] >> 16) & 0xFFFF
    return p[:k_actual].astype(np.float64) / (2**OUT_FRAC)

def quantize_input(logits_f32, k_actual, k_padded):
    q = np.array(logits_f32[:k_actual], dtype=np.float64) * (2**IN_FRAC)
    sat = np.clip(q, -32768, 32767)
    saturated = int(np.sum(np.abs(q) > 32767))
    int16 = np.round(sat).astype(np.int16)
    full = np.full(k_padded, -32768, dtype=np.int16)
    full[:k_actual] = int16
    return full, saturated

def wait_p1_done(timeout_polls=2_000_000):
    for _ in range(timeout_polls):
        if mmio.read(REG_STATUS) & STATUS_P1_DONE:
            return True
    return False

def wait_p2_done(timeout_polls=2_000_000):
    for _ in range(timeout_polls):
        if mmio.read(REG_STATUS) & STATUS_P2_DONE:
            return True
    return False

def read_perf_counters():
    return {
        'total':   int(rd64(REG_TOTAL_LO,   REG_TOTAL_HI)),
        'p1':      int(rd64(REG_P1_LO,      REG_P1_HI)),
        'p2':      int(rd64(REG_P2_LO,      REG_P2_HI)),
        'stall':   int(rd64(REG_STALL_LO,   REG_STALL_HI)),
        'compute': int(rd64(REG_COMPUTE_LO, REG_COMPUTE_HI)),
    }

def run_softmax_v6(logits_f32, k_actual, k_padded, in_buf, out_buf):
    int16, saturated = quantize_input(logits_f32, k_actual, k_padded)
    packed = pack_128b(int16, k_padded)
    np.copyto(in_buf[:len(packed)], packed)

    mmio.write(REG_CTRL, CTRL_CLEAR_PERF)
    # Pass 1: Send data to IP computes max + sum_exp
    mmio.write(REG_CTRL, CTRL_START_P1)
    dma_send.transfer(in_buf); dma_send.wait()
    if not wait_p1_done():
        return None, -1, {'valid': False, 'hw_error': True,
                          'saturated': saturated,
                          'hw_sum': 0, 'timeout': 'P1',
                          'total': 0, 'p1': 0, 'p2': 0,
                          'stall': 0, 'compute': 0}
    # Pass 2: Re-send data to IP outputs probabilities
    dma_recv.transfer(out_buf)
    mmio.write(REG_CTRL, CTRL_START_P2)
    dma_send.transfer(in_buf); dma_send.wait()
    dma_recv.wait()
    probs = unpack_128b(np.array(out_buf), k_actual, k_padded)
    aidx = mmio.read(REG_ARGMAX_IDX)
    status = mmio.read(REG_STATUS)
    hw_err = bool(status & STATUS_ERROR)
    if hw_err: mmio.write(REG_CTRL, CTRL_CLEAR_ERR)
    valid = abs(float(np.sum(probs)) - 1.0) < 0.1
    pred = int(aidx) if aidx < k_actual else int(np.argmax(probs))
    perf = read_perf_counters()
    perf.update({'valid': valid, 'hw_error': hw_err,
                 'saturated': saturated,
                 'hw_sum': float(np.sum(probs))})
    return probs, pred, perf

def generate_scenarios(k, n_per):
    scenarios = []
    for _ in range(n_per):
        scenarios.append(('Uniform_Small',
            np.random.uniform(-2, 2, k).astype(np.float32)))
    for _ in range(n_per):
        scenarios.append(('Uniform_Full',
            np.random.uniform(-7.9, 7.9, k).astype(np.float32)))
    for _ in range(n_per):
        scenarios.append(('Gaussian_s2',
            (np.random.randn(k) * 2).astype(np.float32)))
    for _ in range(n_per):
        v = np.random.uniform(-1, 1, k).astype(np.float32)
        v[np.random.randint(k)] = 7.5
        scenarios.append(('Dominant', v))
    for _ in range(n_per):
        scenarios.append(('Near_Zero',
            np.random.uniform(-0.01, 0.01, k).astype(np.float32)))
    for _ in range(n_per):
        v = np.sort(np.random.uniform(-2, 6, k).astype(np.float32))
        scenarios.append(('Ascending', v))
    for _ in range(n_per):
        v = np.sort(
            np.random.uniform(-2, 6, k).astype(np.float32)
        )[::-1].copy()
        scenarios.append(('Descending', v))
    return scenarios

def evaluate_checks(r):
    return {
        'ConfArgMatch': (
            r['conf_argmatch'] >= THRESH['ConfArgMatch'],
            f"{r['conf_argmatch']:.2f}% >= {THRESH['ConfArgMatch']}% "
            f"({r['conf_match']}/{r['conf_total']} conf, "
            f"{r['ambiguous']} ambig)"),
        'MAE': (r['mae'] <= THRESH['MAE'],
            f"{r['mae']:.6f} <= {THRESH['MAE']}"),
        'Cosine': (r['cos'] >= THRESH['Cosine'],
            f"{r['cos']:.6f} >= {THRESH['Cosine']}"),
        'KL': (r['kl'] <= THRESH['KL'],
            f"{r['kl']:.6f} <= {THRESH['KL']}"),
        'Valid': (r['valid_pct'] >= THRESH['Valid'],
            f"{r['valid_pct']:.2f}% >= {THRESH['Valid']}%"),
        'HW_Errors': (r['errors'] <= THRESH['HW_Errors'],
            f"{r['errors']} = {THRESH['HW_Errors']}"),
    }

print('Functions ready.')

Functions ready.


In [4]:
# ==========================================================================
# debug_argmax_v6.py — Argmax diagnostic on Kria KV260
#
# Run AFTER loading the notebook's Cell 1 + Cell 2 (overlay + functions).
# Paste this into a notebook cell or run it separately.
#
# Tests a few controlled scenarios and prints BOTH SW and HW argmax
# to pinpoint where the mismatch occurs.
# ==========================================================================
import numpy as np
import torch
import torch.nn.functional as F

# --- Small helper to run one sample and print diagnostics ---
def debug_one(label, logits_f32, K, K_PADDED, in_buf, out_buf, mmio):
    """Run one softmax and print argmax diagnostics."""
    int16, sat = quantize_input(logits_f32, K, K_PADDED)
    packed = pack_128b(int16, K_PADDED)
    np.copyto(in_buf[:len(packed)], packed)

    mmio.write(REG_CTRL, CTRL_CLEAR_PERF)

    # Pass 1
    mmio.write(REG_CTRL, CTRL_START_P1)
    dma_send.transfer(in_buf); dma_send.wait()
    if not wait_p1_done():
        print(f"  [{label}] TIMEOUT on P1!")
        return

    # Read argmax AFTER P1 (before P2 overwrites anything)
    hw_argmax_p1 = mmio.read(REG_ARGMAX_IDX)
    hw_argval_p1 = mmio.read(REG_ARGMAX_VAL)

    # Pass 2
    dma_recv.transfer(out_buf)
    mmio.write(REG_CTRL, CTRL_START_P2)
    dma_send.transfer(in_buf); dma_send.wait()
    dma_recv.wait()

    probs = unpack_128b(np.array(out_buf), K, K_PADDED)

    # Read argmax AFTER P2
    hw_argmax_p2 = mmio.read(REG_ARGMAX_IDX)
    hw_argval_p2 = mmio.read(REG_ARGMAX_VAL)

    # SW reference
    psw = F.softmax(torch.tensor(logits_f32[:K]), dim=0).numpy()
    sw_argmax = int(np.argmax(psw))

    # Also compute argmax from raw quantized input
    sw_argmax_raw = int(np.argmax(int16[:K].astype(np.float64)))

    # Find actual max in the quantized input
    max_q_val = int(np.max(int16[:K]))
    max_q_idx = int(np.argmax(int16[:K]))

    # HW argmax_val is Q4.12 signed — interpret
    hw_val_signed = hw_argval_p1
    if hw_val_signed >= 0x8000:
        hw_val_signed = hw_val_signed - 0x10000

    match_p1 = "PASS" if hw_argmax_p1 == sw_argmax else "FAIL"
    match_p2 = "PASS" if hw_argmax_p2 == sw_argmax else "FAIL"
    match_raw = "PASS" if hw_argmax_p1 == sw_argmax_raw else "FAIL"

    print(f"  [{label}]")
    print(f"    SW argmax (softmax):  {sw_argmax}")
    print(f"    SW argmax (raw Q412): {sw_argmax_raw} (max_q_val={max_q_val})")
    print(f"    HW argmax after P1:   {hw_argmax_p1} (val=0x{hw_argval_p1:04X} = {hw_val_signed})")
    print(f"    HW argmax after P2:   {hw_argmax_p2} (val=0x{hw_argval_p2:04X})")
    print(f"    Match(P1 vs SW):      {match_p1}  (raw: {match_raw})")
    print(f"    Match(P2 vs SW):      {match_p2}")
    print(f"    sum(probs)={np.sum(probs):.6f}  MAE={np.mean(np.abs(probs-psw)):.8f}")

    # Check which beat the true max is in
    beat_of_max = sw_argmax_raw // 8
    elem_in_beat = sw_argmax_raw % 8
    print(f"    True max at beat {beat_of_max}, elem {elem_in_beat} "
          f"(block {beat_of_max // 16}, beat-in-block {beat_of_max % 16})")

    # If mismatch, show what HW thinks vs what it should be
    if hw_argmax_p1 != sw_argmax_raw:
        hw_beat = hw_argmax_p1 // 8
        hw_elem = hw_argmax_p1 % 8
        hw_val_at_idx = int(int16[hw_argmax_p1]) if hw_argmax_p1 < K else -99999
        true_val = int(int16[sw_argmax_raw])
        print(f"    MISMATCH: HW says beat {hw_beat} elem {hw_elem} "
              f"(val={hw_val_at_idx}), true max at idx {sw_argmax_raw} (val={true_val})")
    print()


# ============================================================
# RUN DIAGNOSTICS
# ============================================================
K = 1024
K_PADDED = ((K + 7) // 8) * 8
N_BEATS = K_PADDED // 8
BUF_WORDS = N_BEATS * 4

try: in_buf.freebuffer(); out_buf.freebuffer()
except: pass
in_buf  = allocate(shape=(BUF_WORDS,), dtype=np.uint32)
out_buf = allocate(shape=(BUF_WORDS,), dtype=np.uint32)
mmio.write(REG_K_CONFIG, K_PADDED)

print("=" * 70)
print("  ARGMAX DIAGNOSTIC — K=1024")
print("=" * 70)

# Test 1: Ascending — max should be at index K-1
np.random.seed(99)
v_asc = np.sort(np.random.uniform(-2, 6, K).astype(np.float32))
debug_one("Ascending", v_asc, K, K_PADDED, in_buf, out_buf, mmio)

# Test 2: Descending — max should be at index 0
v_desc = v_asc[::-1].copy()
debug_one("Descending", v_desc, K, K_PADDED, in_buf, out_buf, mmio)

# Test 3: Dominant at index 500 — max should be at 500
v_dom = np.random.uniform(-1, 1, K).astype(np.float32)
v_dom[500] = 7.5
debug_one("Dominant@500", v_dom, K, K_PADDED, in_buf, out_buf, mmio)

# Test 4: Dominant at index 7 — first beat, last element
v_dom2 = np.random.uniform(-1, 1, K).astype(np.float32)
v_dom2[7] = 7.5
debug_one("Dominant@7", v_dom2, K, K_PADDED, in_buf, out_buf, mmio)

# Test 5: Dominant at index 0 — first element
v_dom3 = np.random.uniform(-1, 1, K).astype(np.float32)
v_dom3[0] = 7.5
debug_one("Dominant@0", v_dom3, K, K_PADDED, in_buf, out_buf, mmio)

# Test 6: Dominant at index 127 — last element of block 0
v_dom4 = np.random.uniform(-1, 1, K).astype(np.float32)
v_dom4[127] = 7.5
debug_one("Dominant@127", v_dom4, K, K_PADDED, in_buf, out_buf, mmio)

# Test 7: Dominant at index 128 — first element of block 1
v_dom5 = np.random.uniform(-1, 1, K).astype(np.float32)
v_dom5[128] = 7.5
debug_one("Dominant@128", v_dom5, K, K_PADDED, in_buf, out_buf, mmio)

# Test 8: Dominant at index 1023 — last element
v_dom6 = np.random.uniform(-1, 1, K).astype(np.float32)
v_dom6[1023] = 7.5
debug_one("Dominant@1023", v_dom6, K, K_PADDED, in_buf, out_buf, mmio)

# Test 9: All same value — argmax should be 0
v_flat = np.full(K, 1.0, dtype=np.float32)
debug_one("AllSame(1.0)", v_flat, K, K_PADDED, in_buf, out_buf, mmio)

# Test 10: Max at beat boundary (index 15 vs 16)
v_boundary = np.random.uniform(-1, 1, K).astype(np.float32)
v_boundary[120] = 7.5  # last beat of block 0 (beat 15)
debug_one("Dominant@120(beat15)", v_boundary, K, K_PADDED, in_buf, out_buf, mmio)

try: in_buf.freebuffer(); out_buf.freebuffer()
except: pass
print("=" * 70)
print("  DONE")
print("=" * 70)


  ARGMAX DIAGNOSTIC — K=1024
  [Ascending]
    SW argmax (softmax):  1023
    SW argmax (raw Q412): 1023 (max_q_val=24555)
    HW argmax after P1:   1023 (val=0x5FEB = 24555)
    HW argmax after P2:   1023 (val=0x5FEB)
    Match(P1 vs SW):      PASS  (raw: PASS)
    Match(P2 vs SW):      PASS
    sum(probs)=0.985809  MAE=0.00001387
    True max at beat 127, elem 7 (block 7, beat-in-block 15)

  [Descending]
    SW argmax (softmax):  0
    SW argmax (raw Q412): 0 (max_q_val=24555)
    HW argmax after P1:   0 (val=0x5FEB = 24555)
    HW argmax after P2:   0 (val=0x5FEB)
    Match(P1 vs SW):      PASS  (raw: PASS)
    Match(P2 vs SW):      PASS
    sum(probs)=0.985931  MAE=0.00001376
    True max at beat 0, elem 0 (block 0, beat-in-block 0)

  [Dominant@500]
    SW argmax (softmax):  500
    SW argmax (raw Q412): 500 (max_q_val=30720)
    HW argmax after P1:   500 (val=0x7800 = 30720)
    HW argmax after P2:   500 (val=0x7800)
    Match(P1 vs SW):      PASS  (raw: PASS)
    Match(P2 vs SW

In [5]:
# =========================================================================
# Cell 3: Run Benchmark
# =========================================================================
np.random.seed(42)

all_results = {}
max_passed_k = 0
mode_tag = f'{"earlystop" if RUN_MODE == "A" else "fullrun"}'
LOG_CSV = f'pure_softmax_v6_{mode_tag}.csv'

csv_header = [
    'K', 'Scenario', 'Sample', 'ArgmaxOK', 'ConfidentCase',
    'ConfArgOK', 'SW_Gap', 'MAE', 'MaxAE', 'KL', 'Cosine',
    'HW_Sum', 'Valid', 'HW_Error',
    'Compute_Cyc', 'Stall_Cyc', 'P1_Cyc', 'P2_Cyc', 'Total_Cyc']

print(f"\n{'='*90}")
print(f'  PURE SOFTMAX SCALABILITY — IP v6.0 '
      f'(Block-Scan Lookahead) @ {IP_FCLK_HZ/1e6:.0f} MHz')
print(f'  Mode {RUN_MODE}: '
      f'{"Early-Stop" if RUN_MODE == "A" else "Full-Run"}')
print(f'  Pipeline: scan(3) + exp2(6) + csa(1) + booth(9) '
      f'+ shift(1) + clamp(1) = 21 stages')
print(f'  Block-scan: 16-beat lookahead, 3-stage pipelined comparator')
print(f"{'='*90}")
print(f'  K schedule: {K_SCHEDULE}')
print(f'  Per K: 7 scenarios x {N_SAMPLES_PER_SCENARIO} '
      f'= {7*N_SAMPLES_PER_SCENARIO} samples')
print(f"{'='*90}\n")

with open(LOG_CSV, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(csv_header)

    for K in K_SCHEDULE:
        K_PADDED = ((K + 7) // 8) * 8
        N_BEATS = K_PADDED // 8
        BUF_WORDS = N_BEATS * 4

        try: in_buf.freebuffer(); out_buf.freebuffer()
        except: pass
        in_buf  = allocate(shape=(BUF_WORDS,), dtype=np.uint32)
        out_buf = allocate(shape=(BUF_WORDS,), dtype=np.uint32)
        mmio.write(REG_K_CONFIG, K_PADDED)

        print(f"{'='*90}")
        print(f"  K = {K:,} ({N_BEATS} beats, "
              f"{K_PADDED*2:,} bytes)")
        print(f"{'='*90}")

        scenarios = generate_scenarios(K, N_SAMPLES_PER_SCENARIO)
        total_n = len(scenarios)

        # Accumulators
        raw_match = conf_match_cnt = conf_total = 0
        ambiguous_cnt = 0
        mae_sum = maxae_sum = kl_sum = cos_sum = 0.0
        valid_cnt = error_cnt = sat_cnt = 0
        compute_sum = stall_sum = 0
        p1_sum = p2_sum = total_sum = 0
        hw_sum_list = []
        scenario_metrics = {}

        t0 = time.time()

        for idx, (sname, logits) in enumerate(scenarios):
            psw = F.softmax(torch.tensor(logits), dim=0).numpy()
            sw_argmax = int(np.argmax(psw))
            sorted_sw = np.sort(psw)[::-1]
            sw_gap = float(sorted_sw[0] - sorted_sw[1])
            is_confident = sw_gap > CONF_GAP_THRESH

            phw, hw_argmax, info = run_softmax_v6(
                logits, K, K_PADDED, in_buf, out_buf)

            if phw is None:  # timeout
                error_cnt += 1
                print(f'       Timeout at '
                      f'sample {idx} ({sname})')
                continue

            raw_ok = int(hw_argmax == sw_argmax)
            raw_match += raw_ok
            if is_confident:
                conf_total += 1
                conf_match_cnt += raw_ok
            else:
                ambiguous_cnt += 1

            mae = float(np.mean(np.abs(phw - psw)))
            maxae = float(np.max(np.abs(phw - psw)))
            kl = kl_div(psw, phw)
            cs = cos_sim(psw, phw)

            mae_sum += mae; maxae_sum += maxae
            kl_sum += kl; cos_sum += cs
            if info['valid']: valid_cnt += 1
            if info['hw_error']: error_cnt += 1
            if info['saturated'] > 0: sat_cnt += 1
            compute_sum += info['compute']
            stall_sum += info['stall']
            p1_sum += info['p1']
            p2_sum += info['p2']
            total_sum += info['total']
            hw_sum_list.append(info['hw_sum'])

            if sname not in scenario_metrics:
                scenario_metrics[sname] = {
                    'n':0, 'raw_match':0, 'conf_match':0,
                    'conf_total':0, 'ambiguous':0,
                    'mae':0, 'kl':0, 'cos':0,
                    'compute':0, 'stall':0,
                    'valid':0, 'hw_sum_err':0
                }
            sm = scenario_metrics[sname]
            sm['n'] += 1
            sm['raw_match'] += raw_ok
            sm['mae'] += mae
            sm['kl'] += kl; sm['cos'] += cs
            sm['compute'] += info['compute']
            sm['stall'] += info['stall']
            if info['valid']: sm['valid'] += 1
            sm['hw_sum_err'] += abs(info['hw_sum'] - 1.0)
            if is_confident:
                sm['conf_total'] += 1
                sm['conf_match'] += raw_ok
            else:
                sm['ambiguous'] += 1

            writer.writerow([
                K, sname, idx, raw_ok,
                int(is_confident),
                int(raw_ok) if is_confident else 'N/A',
                f'{sw_gap:.8f}', f'{mae:.8f}',
                f'{maxae:.8f}', f'{kl:.8f}',
                f'{cs:.8f}',
                f'{info["hw_sum"]:.6f}',
                info['valid'], info['hw_error'],
                info['compute'], info['stall'],
                info['p1'], info['p2'],
                info['total']])

            if (idx + 1) % (total_n // 4) == 0:
                c_pct = (conf_match_cnt
                         / max(conf_total, 1) * 100)
                print(
                    f"    [{idx+1}/{total_n}] "
                    f"ConfMatch={c_pct:.1f}% "
                    f"Raw={raw_match/(idx+1)*100:.1f}% "
                    f"MAE={mae_sum/(idx+1):.6f} "
                    f"Valid={valid_cnt/(idx+1)*100:.1f}%")

        elapsed = time.time() - t0
        n = total_n

        # Aggregate
        avg_mae = mae_sum / n
        avg_kl = kl_sum / n
        avg_cos = cos_sum / n
        raw_match_pct = raw_match / n * 100
        conf_match_pct = (conf_match_cnt
                          / max(conf_total, 1) * 100)
        valid_pct = valid_cnt / n * 100
        avg_compute = compute_sum / n
        avg_stall = stall_sum / n
        avg_total = total_sum / n
        compute_gbps = (K_PADDED * 16 * IP_FCLK_HZ
                        / max(avg_compute, 1)) / 1e9
        compute_lat_us = avg_compute / IP_FCLK_HZ * 1e6
        ip_fps = IP_FCLK_HZ / max(avg_compute, 1)
        hw_sum_arr = np.array(hw_sum_list)
        avg_sum_err = float(
            np.mean(np.abs(hw_sum_arr - 1.0)))

        r = {
            'conf_argmatch': conf_match_pct,
            'raw_argmatch': raw_match_pct,
            'conf_match': conf_match_cnt,
            'conf_total': conf_total,
            'ambiguous': ambiguous_cnt,
            'mae': avg_mae,
            'maxae': maxae_sum / n,
            'kl': avg_kl, 'cos': avg_cos,
            'valid_pct': valid_pct,
            'errors': error_cnt,
            'saturated': sat_cnt,
            'avg_compute': avg_compute,
            'avg_stall': avg_stall,
            'avg_p1': p1_sum / n,
            'avg_p2': p2_sum / n,
            'avg_total': avg_total,
            'compute_gbps': compute_gbps,
            'compute_lat_us': compute_lat_us,
            'ip_fps': ip_fps,
            'avg_sum_err': avg_sum_err,
            'elapsed': elapsed,
            'n_samples': n,
            'scenario_metrics': scenario_metrics,
        }

        checks = evaluate_checks(r)
        all_passed = all(v[0] for v in checks.values())
        r['passed'] = all_passed
        all_results[K] = r

        status = 'PASS' if all_passed else 'FAIL'
        print(f"\n  K={K:>6,} | {status}")
        for name, (passed, desc) in checks.items():
            print(f"    {name:<14s}: {desc}")
        print(f"    Raw ArgMatch:  "
              f"{raw_match_pct:.2f}% (incl. ambiguous)")
        print(f"    Avg |sum-1.0|: {avg_sum_err:.6f}")
        stall_pct = (avg_stall
                     / max(avg_compute, 1) * 100)
        print(
            f"    Compute: {avg_compute:.0f} cyc "
            f"({compute_lat_us:.2f} us) | "
            f"Stall: {avg_stall:.0f} "
            f"({stall_pct:.1f}%) | "
            f"BW: {compute_gbps:.2f} Gbps | "
            f"FPS: {ip_fps:.0f}")
        print(f"    Wall: {elapsed:.1f}s | "
              f"E2E FPS: {n/elapsed:.0f}")

        # Per-scenario table
        print(f"\n    {'Scenario':<16s} "
              f"{'RawArg':>7s} {'ConfArg':>8s} "
              f"{'Ambig':>5s} {'MAE':>10s} "
              f"{'KL':>10s} {'Cos':>10s} "
              f"{'Valid':>6s} {'Compute':>7s} "
              f"{'Stall':>6s} {'Stall%':>6s}")
        print(f"    {'='*16} {'='*7} {'='*8} "
              f"{'='*5} {'='*10} {'='*10} "
              f"{'='*10} {'='*6} {'='*7} "
              f"{'='*6} {'='*6}")
        for sname in sorted(scenario_metrics.keys()):
            sm = scenario_metrics[sname]
            sn = sm['n']; ct = sm['conf_total']
            ca = (f"{sm['conf_match']/ct*100:.1f}%"
                  if ct > 0 else 'N/A')
            sc = sm['compute']/sn
            ss = sm['stall']/sn
            sp = ss / max(sc, 1) * 100
            print(
                f"    {sname:<16s} "
                f"{sm['raw_match']/sn*100:>6.1f}% "
                f"{ca:>8s} {sm['ambiguous']:>5d} "
                f"{sm['mae']/sn:>10.6f} "
                f"{sm['kl']/sn:>10.6f} "
                f"{sm['cos']/sn:>10.6f} "
                f"{sm['valid']/sn*100:>5.1f}% "
                f"{sc:>7.0f} {ss:>6.0f} "
                f"{sp:>5.1f}%")

        if all_passed:
            max_passed_k = K

        # Mode A: early-stop
        if RUN_MODE == 'A' and not all_passed:
            print(f"\n  [Mode A] Stopping. "
                  f"Max passing K = {max_passed_k:,}")
            break
        elif all_passed:
            tag = ('[Mode A]' if RUN_MODE == 'A'
                   else '[Mode B]')
            print(f"\n  {tag} PASSED. Continuing...")
        else:
            print(f"\n  [Mode B] FAILED but "
                  f"continuing (full-run mode)...")

        in_buf.freebuffer()
        out_buf.freebuffer()

print(f"\n{'='*90}")
print(f'  Mode {RUN_MODE} COMPLETE | '
      f'Max passing K = {max_passed_k:,}')
print(f'  CSV: {LOG_CSV}')
print(f"{'='*90}")


  PURE SOFTMAX SCALABILITY — IP v6.0 (Block-Scan Lookahead) @ 333 MHz
  Mode A: Early-Stop
  Pipeline: scan(3) + exp2(6) + csa(1) + booth(9) + shift(1) + clamp(1) = 21 stages
  Block-scan: 16-beat lookahead, 3-stage pipelined comparator
  K schedule: [1024, 2048, 4096, 8192, 16384, 32768]
  Per K: 7 scenarios x 200 = 1400 samples

  K = 1,024 (128 beats, 2,048 bytes)
    [350/1400] ConfMatch=100.0% Raw=98.9% MAE=0.000013 Valid=100.0%
    [700/1400] ConfMatch=100.0% Raw=99.4% MAE=0.000013 Valid=100.0%
    [1050/1400] ConfMatch=100.0% Raw=83.7% MAE=0.000014 Valid=100.0%
    [1400/1400] ConfMatch=100.0% Raw=87.6% MAE=0.000014 Valid=100.0%

  K= 1,024 | PASS
    ConfArgMatch  : 100.00% >= 99.0% (685/685 conf, 715 ambig)
    MAE           : 0.000014 <= 0.001
    Cosine        : 0.999970 >= 0.999
    KL            : 0.039542 <= 0.5
    Valid         : 100.00% >= 99.0%
    HW_Errors     : 0 = 0
    Raw ArgMatch:  87.64% (incl. ambiguous)
    Avg |sum-1.0|: 0.012269
    Compute: 330 cyc (0.99

In [6]:
# =========================================================================
# Cell 4: Summary Report (UPDATED - TABLE NGAY NGẮN & CÂN ĐỐI)
# =========================================================================

ks = sorted(all_results.keys())

# ====================== CẤU HÌNH CỘT ======================
COLS = [
    ("K",          9,  '>'),   # K value with comma
    ("St",         5,  '>'),
    ("ConfArg",    9,  '>'),
    ("RawArg",     9,  '>'),
    ("Ambig",      6,  '>'),
    ("MAE",       11,  '>'),
    ("KL",        11,  '>'),
    ("Cos",       11,  '>'),
    ("Valid",      8,  '>'),
    ("|S-1|",      9,  '>'),
    ("Comp",       8,  '>'),
    ("Stall",      7,  '>'),
    ("St%",        6,  '>'),
    ("BW",         9,  '>'),
    ("Lat",        9,  '>'),
    ("FPS",       11,  '>'),
]

def print_header():
    header = " ".join(f"{name:^{w}}" if a=='^' else f"{name:{a}{w}}" 
                     for name, w, a in COLS)
    sep    = " ".join("─" * w for _, w, _ in COLS)
    print(header)
    print(sep)

def print_row(K, r):
    sp = (r['avg_stall'] / max(r['avg_compute'], 1) * 100)
    values = [
        f"{K:,}",
        'PASS' if r['passed'] else 'FAIL',
        f"{r['conf_argmatch']:7.2f}%",
        f"{r['raw_argmatch']:7.2f}%",
        f"{r['ambiguous']:5d}",
        f"{r['mae']:10.6f}",
        f"{r['kl']:10.6f}",
        f"{r['cos']:10.6f}",
        f"{r['valid_pct']:6.2f}%",
        f"{r['avg_sum_err']:8.5f}",
        f"{r['avg_compute']:7.0f}",
        f"{r['avg_stall']:6.0f}",
        f"{sp:4.1f}%",
        f"{r['compute_gbps']:7.2f}G",
        f"{r['compute_lat_us']:7.2f}us",
        f"{r['ip_fps']:10.0f}",
    ]
    line = " ".join(f"{val:{a}{w}}" for (name,w,a), val in zip(COLS, values))
    print(line)

# ====================== REPORT ======================
total_width = sum(w + 1 for _, w, _ in COLS) - 1   # +1 cho khoảng trắng

print(f"\n{'═' * total_width}")
print(f' PURE SOFTMAX v6.0 (Block-Scan) — Mode {RUN_MODE} REPORT')
print(f' PAR8, 128-bit AXI-S, {IP_FCLK_HZ/1e6:.0f} MHz, '
      f'0 DSP, 2 BRAM36K, auto-P2')
print(f"{'═' * total_width}")

print_header()

for K in ks:
    r = all_results[K]
    print_row(K, r)

print(f"\n Max passing K: {max_passed_k:,}")

# ====================== SCALING TRENDS ======================
if len(ks) >= 2:
    print(f"\n{'─' * total_width}")
    print(f" --- SCALING TRENDS ---")
    print(f" {'K1->K2':>12} {'Comp x':>8} {'Stall x':>8} {'BW chg':>8} "
          f"{'KL chg':>9} {'Cos chg':>11} {'Valid chg':>10}")
    print(f" {'─'*12} {'─'*8} {'─'*8} {'─'*8} {'─'*9} {'─'*11} {'─'*10}")
    
    for i in range(1, len(ks)):
        k1, k2 = ks[i-1], ks[i]
        r1, r2 = all_results[k1], all_results[k2]
        
        comp_x  = r2['avg_compute'] / max(r1['avg_compute'], 1)
        stall_x = r2['avg_stall']   / max(r1['avg_stall'],   1)
        bw_chg  = r2['compute_gbps'] - r1['compute_gbps']
        kl_chg  = r2['kl'] - r1['kl']
        cos_chg = r2['cos'] - r1['cos']
        val_chg = r2['valid_pct'] - r1['valid_pct']
        
        print(f" {k1:>5,}->{k2:<5,} "
              f"{comp_x:>7.2f}x {stall_x:>7.2f}x "
              f"{bw_chg:>+7.2f}G {kl_chg:>+8.4f} "
              f"{cos_chg:>+10.6f} {val_chg:>+9.2f}%")

print(f"\n{'═' * total_width}")


═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 PURE SOFTMAX v6.0 (Block-Scan) — Mode A REPORT
 PAR8, 128-bit AXI-S, 333 MHz, 0 DSP, 2 BRAM36K, auto-P2
═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
        K    St   ConfArg    RawArg  Ambig         MAE          KL         Cos    Valid     |S-1|     Comp   Stall    St%        BW       Lat         FPS
───────── ───── ───────── ───────── ────── ─────────── ─────────── ─────────── ──────── ───────── ──────── ─────── ────── ───────── ───────── ───────────
    1,024  PASS   100.00%    87.64%    715    0.000014    0.039542    0.999970  100.00%   0.01227      330      33  10.1%    16.55G    0.99us     1010134
    2,048  PASS   100.00%    85.64%    925    0.000015    0.085388    0.999484   99.93%   0.02947      610      57   9.4%   

In [7]:
# =========================================================================
# Cell 5: Per-Scenario Breakdown
# =========================================================================
print(f"\n{'='*105}")
print(f'  WORST-CASE SCENARIO ANALYSIS')
print(f"{'='*105}")

for K in ks:
    r = all_results[K]
    sm = r['scenario_metrics']
    print(f"\n  K = {K:,}")
    print(f"  {'Scenario':<16s} {'RawArg':>7s} {'ConfArg':>8s} {'Ambi':>5s} "
          f"{'MAE':>10s} {'KL':>10s} {'Cos':>10s} {'Valid':>6s} {'Stall%':>6s}")
    print(f"  {'='*16} {'='*7} {'='*8} {'='*5} "
          f"{'='*10} {'='*10} {'='*10} {'='*6} {'='*6}")

    worst_s = None; worst_cos = 1.0
    for sname in sorted(sm.keys()):
        s = sm[sname]; sn = s['n']; ct = s['conf_total']
        ca = f"{s['conf_match']/ct*100:.1f}%" if ct > 0 else 'N/A'
        sc = s['compute']/sn; ss = s['stall']/sn; sp = ss/max(sc,1)*100
        s_cos = s['cos']/sn
        print(f"  {sname:<16s} {s['raw_match']/sn*100:>6.1f}% {ca:>8s} {s['ambiguous']:>5d} "
              f"{s['mae']/sn:>10.6f} {s['kl']/sn:>10.6f} {s_cos:>10.6f} "
              f"{s['valid']/sn*100:>5.1f}% {sp:>5.1f}%")
        if s_cos < worst_cos:
            worst_cos = s_cos; worst_s = sname

    total_ambig = sum(s['ambiguous'] for s in sm.values())
    print(f"  -> Worst (Cos): {worst_s} ({worst_cos:.6f})")
    if total_ambig > 0:
        print(f"  -> Ambiguous: {total_ambig}/{r['n_samples']} ({total_ambig/r['n_samples']*100:.1f}%)")

print(f"\n{'='*105}")


  WORST-CASE SCENARIO ANALYSIS

  K = 1,024
  Scenario          RawArg  ConfArg  Ambi        MAE         KL        Cos  Valid Stall%
  ================ ======= ======== ===== ========== ========== ========== ====== ======
  Ascending          99.0%   100.0%   134   0.000014   0.070819   0.999973 100.0%  27.4%
  Descending        100.0%   100.0%   133   0.000014   0.069501   0.999973 100.0%   0.0%
  Dominant          100.0%   100.0%     0   0.000013   0.004398   1.000000 100.0%   9.1%
  Gaussian_s2       100.0%   100.0%     1   0.000016   0.061729   0.999967 100.0%   8.7%
  Near_Zero          16.5%      N/A   200   0.000014   0.014454   0.999937 100.0%   2.8%
  Uniform_Full      100.0%   100.0%    49   0.000008   0.038788   0.999992 100.0%   8.4%
  Uniform_Small      98.0%   100.0%   198   0.000016   0.017103   0.999949 100.0%   8.5%
  -> Worst (Cos): Near_Zero (0.999937)
  -> Ambiguous: 715/1400 (51.1%)

  K = 2,048
  Scenario          RawArg  ConfArg  Ambi        MAE         KL      

In [8]:
# =========================================================================
# Cell 8: Cleanup
# =========================================================================
try: in_buf.freebuffer(); out_buf.freebuffer()
except: pass
print('Buffers freed. Done.')

Buffers freed. Done.
